# Fase 1 – Modelo Predictivo
## Student Performance & Study Habits Dataset

**Integrantes:**
- Brahian Ocampo Garcia
- Carlos Andres Pineda Ospina

**Curso:** Modelos y Simulación de Sistemas I — Andrés Parra — 2026-2

**Objetivo:** Construir un modelo de regresión para predecir el puntaje del examen final (`final_exam_score`) a partir de hábitos de estudio y factores demográficos.

**Dataset:** [Student Performance & Study Habits (Kaggle)](https://www.kaggle.com/datasets/harshadapatil31/student-performance-and-study-habits-dataset)

---

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib
import os

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Librerías cargadas correctamente.")

## 2. Carga y exploración inicial de los datos

> **Nota:** El archivo `student_performance_dataset.csv` debe estar en la misma carpeta que este notebook.  
> Descárgalo desde [Kaggle](https://www.kaggle.com/datasets/harshadapatil31/student-performance-and-study-habits-dataset).

In [ ]:
CSV_PATH = "student_performance_dataset.csv"

df = pd.read_csv(CSV_PATH)
print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head(10)

In [ ]:
# Tipos de datos y valores no nulos
df.info()

In [ ]:
# Estadísticas descriptivas — variables numéricas
df.describe().round(2)

In [ ]:
# Estadísticas descriptivas — variables categóricas
df.describe(include='object')

In [ ]:
# Columnas del dataset
print("Columnas del dataset:")
for col in df.columns:
    print(f"  {col:30s} | tipo: {str(df[col].dtype):10s} | nulos: {df[col].isnull().sum()}")

## 3. Análisis de valores faltantes

In [ ]:
# Conteo y porcentaje de valores nulos por columna
nulos = df.isnull().sum()
porcentaje = (nulos / len(df)) * 100
tabla_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': porcentaje.round(2)})
tabla_nulos = tabla_nulos[tabla_nulos['Nulos'] > 0].sort_values('Porcentaje (%)', ascending=False)

if len(tabla_nulos) > 0:
    print("Variables con valores faltantes:")
    print(tabla_nulos)
    print(f"\nTotal de registros con al menos un nulo: {df.isnull().any(axis=1).sum()}")
else:
    print("No hay valores faltantes en el dataset.")

**Observación:** La variable `parental_education` tiene 102 valores faltantes (10.2% del dataset). Aunque este porcentaje supera el rango sugerido (0.1%–2%), es manejable y será tratado mediante imputación con la moda durante el preprocesamiento.

## 4. Análisis Exploratorio de Datos (EDA)

### 4.1 Distribución de la variable objetivo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df['final_exam_score'], bins=25, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Puntaje del examen final')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de final_exam_score')
axes[0].axvline(df['final_exam_score'].mean(), color='red', linestyle='--',
                label=f"Media: {df['final_exam_score'].mean():.1f}")
axes[0].axvline(df['final_exam_score'].median(), color='orange', linestyle='--',
                label=f"Mediana: {df['final_exam_score'].median():.1f}")
axes[0].legend()

# Boxplot
axes[1].boxplot(df['final_exam_score'], vert=True)
axes[1].set_ylabel('Puntaje del examen final')
axes[1].set_title('Boxplot de final_exam_score')

plt.tight_layout()
plt.show()

print(f"Media:      {df['final_exam_score'].mean():.2f}")
print(f"Mediana:    {df['final_exam_score'].median():.2f}")
print(f"Desv. est.: {df['final_exam_score'].std():.2f}")
print(f"Mínimo:     {df['final_exam_score'].min()}")
print(f"Máximo:     {df['final_exam_score'].max()}")

### 4.2 Distribución de variables numéricas predictoras

In [ ]:
# Variables numéricas predictoras (sin student_id ni final_exam_score)
cols_numericas = ['study_time_hours', 'attendance_percent', 'sleep_hours', 'previous_grade']

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, col in zip(axes, cols_numericas):
    ax.hist(df[col].dropna(), bins=20, edgecolor='black', alpha=0.7, color='teal')
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

### 4.3 Distribución de variables categóricas

In [ ]:
cols_categoricas = ['gender', 'parental_education', 'internet_access',
                    'extracurricular_activities', 'part_time_job']

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, col in zip(axes, cols_categoricas):
    conteos = df[col].value_counts()
    ax.bar(conteos.index.astype(str), conteos.values, color='coral', edgecolor='black', alpha=0.8)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 4.4 Matriz de correlación

In [ ]:
cols_corr = cols_numericas + ['final_exam_score']
corr_matrix = df[cols_corr].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True)
plt.title('Matriz de correlación — variables numéricas')
plt.tight_layout()
plt.show()

print("\nCorrelación con final_exam_score:")
print(corr_matrix['final_exam_score'].drop('final_exam_score').sort_values(ascending=False).round(4))

### 4.5 Relación entre predictores numéricos y la variable objetivo

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, col in zip(axes, cols_numericas):
    ax.scatter(df[col], df['final_exam_score'], alpha=0.3, s=15, color='steelblue')
    ax.set_xlabel(col)
    ax.set_ylabel('final_exam_score')
    ax.set_title(f'{col} vs final_exam_score')

plt.tight_layout()
plt.show()

### 4.6 Variables categóricas vs variable objetivo

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 5))

for ax, col in zip(axes, cols_categoricas):
    df.boxplot(column='final_exam_score', by=col, ax=ax)
    ax.set_title(f'final_exam_score por {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('final_exam_score')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('')
plt.tight_layout()
plt.show()

### 4.7 Distribución de calificaciones (final_grade)

In [ ]:
conteo_grades = df['final_grade'].value_counts().sort_index()

plt.figure(figsize=(6, 4))
colores = {'A': '#2ecc71', 'B': '#3498db', 'C': '#f39c12', 'D': '#e74c3c', 'F': '#8e44ad'}
plt.bar(conteo_grades.index, conteo_grades.values,
        color=[colores.get(g, 'gray') for g in conteo_grades.index],
        edgecolor='black', alpha=0.85)
plt.xlabel('Calificación')
plt.ylabel('Cantidad de estudiantes')
plt.title('Distribución de calificaciones (final_grade)')
plt.tight_layout()
plt.show()

print(conteo_grades)

## 5. Preparación de los datos

### 5.1 Definir variables predictoras y variable objetivo

**Prevención de fuga de información:**
- Se excluye `final_grade` porque se deriva directamente de `final_exam_score` (es la nota en letras del mismo examen). Incluirla causaría **data leakage**.
- Se excluye `student_id` porque es solo un identificador sin valor predictivo.
- Se conserva `previous_grade` porque es una calificación **previa**, no derivada del target.

In [ ]:
# Definir predictoras y target
cols_a_excluir = ['student_id', 'final_exam_score', 'final_grade']
feature_cols = [c for c in df.columns if c not in cols_a_excluir]

X = df[feature_cols].copy()
y = df['final_exam_score'].copy()

print(f"Variables predictoras ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  - {col:30s} ({X[col].dtype})")
print(f"\nVariable objetivo: final_exam_score")
print(f"\nForma de X: {X.shape}")
print(f"Forma de y: {y.shape}")

### 5.2 División train/test

Dividimos los datos **antes** de cualquier preprocesamiento para evitar fuga de información. El preprocesamiento (imputación, escalado, encoding) se ajustará únicamente con los datos de entrenamiento.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Conjunto de entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Conjunto de prueba:        {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nNulos en train (parental_education): {X_train['parental_education'].isnull().sum()}")
print(f"Nulos en test  (parental_education): {X_test['parental_education'].isnull().sum()}")

### 5.3 Pipeline de preprocesamiento

Usamos `ColumnTransformer` de scikit-learn para construir un pipeline que:
1. **Variables numéricas:** imputa faltantes con la mediana y escala con `StandardScaler`.
2. **Variables categóricas:** imputa faltantes con la moda y aplica `OneHotEncoder`.

El pipeline garantiza que **no hay fuga de información**, ya que `.fit()` se ejecuta solo sobre los datos de entrenamiento.

In [ ]:
# Identificar columnas numéricas y categóricas
num_features = ['study_time_hours', 'attendance_percent', 'sleep_hours', 'previous_grade']
cat_features = ['gender', 'parental_education', 'internet_access',
                'extracurricular_activities', 'part_time_job']

print(f"Features numéricas ({len(num_features)}): {num_features}")
print(f"Features categóricas ({len(cat_features)}): {cat_features}")

# Pipeline para numéricas
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para categóricas
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combinar en ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ]
)

print("\nPipeline de preprocesamiento creado correctamente.")

In [ ]:
# Verificar que el preprocesamiento funciona
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Obtener nombres de features resultantes
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
feature_names = num_features + list(ohe.get_feature_names_out(cat_features))

print(f"Dimensiones después del preprocesamiento:")
print(f"  X_train: {X_train_transformed.shape}")
print(f"  X_test:  {X_test_transformed.shape}")
print(f"\nFeatures resultantes ({len(feature_names)}):")
for i, name in enumerate(feature_names):
    print(f"  {i+1:2d}. {name}")
print(f"\nNaN en train: {np.isnan(X_train_transformed).sum()}")
print(f"NaN en test:  {np.isnan(X_test_transformed).sum()}")

## 6. Entrenamiento y comparación de modelos

Evaluamos 6 modelos de regresión con validación cruzada (5-fold) sobre el conjunto de entrenamiento.

In [ ]:
modelos = {
    'Linear Regression': LinearRegression(),
    'Ridge':             Ridge(alpha=1.0),
    'Lasso':             Lasso(alpha=0.1),
    'Decision Tree':     DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest':     RandomForestRegressor(n_estimators=100, max_depth=15,
                                               random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5,
                                                    random_state=42),
}

resultados = {}

print(f"{'Modelo':25s}  {'R² (CV)':>15s}  {'MAE (CV)':>15s}")
print("-" * 60)

for nombre, modelo in modelos.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', modelo)
    ])

    scores_r2 = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
    scores_mae = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')

    resultados[nombre] = {
        'R2_mean': scores_r2.mean(),
        'R2_std': scores_r2.std(),
        'MAE_mean': -scores_mae.mean(),
        'MAE_std': scores_mae.std(),
    }

    print(f"{nombre:25s}  {scores_r2.mean():.4f} (±{scores_r2.std():.4f})  {-scores_mae.mean():.4f} (±{scores_mae.std():.4f})")

# Tabla resumen ordenada
df_resultados = pd.DataFrame(resultados).T.sort_values('R2_mean', ascending=False)
print("\n=== Ranking por R² ===")
print(df_resultados[['R2_mean', 'MAE_mean']].round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

nombres = df_resultados.index
r2_vals = df_resultados['R2_mean']
mae_vals = df_resultados['MAE_mean']

axes[0].barh(nombres, r2_vals, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('R² (validación cruzada)')
axes[0].set_title('Comparación de modelos — R²')

axes[1].barh(nombres, mae_vals, color='coral', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('MAE (validación cruzada)')
axes[1].set_title('Comparación de modelos — MAE')

plt.tight_layout()
plt.show()

## 7. Modelo seleccionado — Entrenamiento final

In [ ]:
# Seleccionar el mejor modelo según R²
mejor_nombre = df_resultados.index[0]
mejor_modelo = modelos[mejor_nombre]

print(f"Modelo seleccionado: {mejor_nombre}")
print(f"R² promedio en CV:   {df_resultados.loc[mejor_nombre, 'R2_mean']:.4f}")
print(f"MAE promedio en CV:  {df_resultados.loc[mejor_nombre, 'MAE_mean']:.4f}")

# Pipeline final
pipeline_final = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', mejor_modelo)
])

# Entrenar con todo el conjunto de entrenamiento
pipeline_final.fit(X_train, y_train)
print(f"\nModelo entrenado con {X_train.shape[0]} muestras.")

## 8. Evaluación en el conjunto de prueba

In [ ]:
y_pred = pipeline_final.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print("=" * 50)
print("   EVALUACIÓN EN EL CONJUNTO DE PRUEBA")
print("=" * 50)
print(f"  MAE  (Error Absoluto Medio):        {mae:.4f}")
print(f"  MSE  (Error Cuadrático Medio):       {mse:.4f}")
print(f"  RMSE (Raíz del Error Cuadrático):    {rmse:.4f}")
print(f"  R²   (Coeficiente de determinación): {r2:.4f}")
print("=" * 50)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Real vs Predicho
axes[0].scatter(y_test, y_pred, alpha=0.5, s=20, color='steelblue')
lim_min = min(y_test.min(), y_pred.min()) - 5
lim_max = max(y_test.max(), y_pred.max()) + 5
axes[0].plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=2, label='Predicción perfecta')
axes[0].set_xlabel('Valor real')
axes[0].set_ylabel('Valor predicho')
axes[0].set_title('Real vs Predicho')
axes[0].legend()

# 2. Distribución de residuos
residuos = y_test.values - y_pred
axes[1].hist(residuos, bins=25, edgecolor='black', alpha=0.7, color='teal')
axes[1].set_xlabel('Residuo (real - predicho)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de residuos')
axes[1].axvline(0, color='red', linestyle='--')

# 3. Residuos vs Predicho
axes[2].scatter(y_pred, residuos, alpha=0.5, s=20, color='coral')
axes[2].axhline(0, color='red', linestyle='--')
axes[2].set_xlabel('Valor predicho')
axes[2].set_ylabel('Residuo')
axes[2].set_title('Residuos vs Predicho')

plt.tight_layout()
plt.show()

## 9. Importancia de características

In [ ]:
modelo_entrenado = pipeline_final.named_steps['model']

# Obtener nombres de features transformadas
ohe_final = pipeline_final.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
feature_names_final = num_features + list(ohe_final.get_feature_names_out(cat_features))

if hasattr(modelo_entrenado, 'feature_importances_'):
    importancias = modelo_entrenado.feature_importances_

    df_importancia = pd.DataFrame({
        'Feature': feature_names_final,
        'Importancia': importancias
    }).sort_values('Importancia', ascending=True)

    plt.figure(figsize=(10, max(6, len(df_importancia) * 0.35)))
    plt.barh(df_importancia['Feature'], df_importancia['Importancia'],
             color='steelblue', edgecolor='black')
    plt.xlabel('Importancia')
    plt.title(f'Importancia de características — {mejor_nombre}')
    plt.tight_layout()
    plt.show()

    print("\nTop 5 características más importantes:")
    top5 = df_importancia.tail(5).iloc[::-1]
    for _, row in top5.iterrows():
        print(f"  {row['Feature']:35s}  {row['Importancia']:.4f}")

elif hasattr(modelo_entrenado, 'coef_'):
    coefs = modelo_entrenado.coef_
    df_coefs = pd.DataFrame({
        'Feature': feature_names_final,
        'Coeficiente': coefs
    }).sort_values('Coeficiente', key=abs, ascending=True)

    plt.figure(figsize=(10, max(6, len(df_coefs) * 0.35)))
    colores = ['coral' if c < 0 else 'steelblue' for c in df_coefs['Coeficiente']]
    plt.barh(df_coefs['Feature'], df_coefs['Coeficiente'], color=colores, edgecolor='black')
    plt.xlabel('Coeficiente')
    plt.title(f'Coeficientes del modelo — {mejor_nombre}')
    plt.tight_layout()
    plt.show()
else:
    print("El modelo seleccionado no proporciona importancia de características directamente.")

## 10. Guardar el modelo

Guardamos el pipeline completo (preprocesamiento + modelo) con `joblib` para reutilizarlo en las fases 2, 3 y 4 del proyecto.

In [ ]:
os.makedirs('modelo', exist_ok=True)

modelo_path = 'modelo/pipeline_modelo.joblib'
joblib.dump(pipeline_final, modelo_path)

file_size = os.path.getsize(modelo_path) / 1024
print(f"Modelo guardado en: {modelo_path}")
print(f"Tamaño del archivo: {file_size:.1f} KB")

### Verificación: cargar y usar el modelo guardado

In [ ]:
# Cargar el modelo
pipeline_cargado = joblib.load(modelo_path)

# Verificar que produce las mismas predicciones
y_pred_check = pipeline_cargado.predict(X_test)
diferencia = np.abs(y_pred - y_pred_check).max()
print(f"Diferencia máxima entre predicciones: {diferencia}")
print(f"Verificación: {'✅ OK — Modelo cargado correctamente' if diferencia < 1e-10 else '❌ ERROR'}")

# Ejemplo con datos nuevos
print("\n--- Ejemplo de predicción con datos nuevos ---")
ejemplo = pd.DataFrame({
    'study_time_hours': [5.0],
    'attendance_percent': [90.0],
    'sleep_hours': [7.0],
    'previous_grade': [80.0],
    'gender': ['Female'],
    'parental_education': ['Bachelors'],
    'internet_access': ['Yes'],
    'extracurricular_activities': ['Yes'],
    'part_time_job': ['No']
})
pred = pipeline_cargado.predict(ejemplo)
print(f"Estudiante de ejemplo → Puntaje predicho: {pred[0]:.1f}")

## 11. Resumen y conclusiones

### Resumen del proceso

| Paso | Descripción |
|------|-------------|
| **Dataset** | 1000 estudiantes, 9 variables predictoras, fuente: Kaggle |
| **Variable objetivo** | `final_exam_score` (regresión, valor entre 0 y 100) |
| **Valores faltantes** | `parental_education` (102 nulos, 10.2%) → imputados con la moda |
| **Fuga de información** | Se excluyó `final_grade` (derivada del target); split train/test antes de preprocesar |
| **Preprocesamiento** | Pipeline con `ColumnTransformer`: escalado + one-hot encoding |
| **Modelos evaluados** | Linear Regression, Ridge, Lasso, Decision Tree, Random Forest, Gradient Boosting |
| **Evaluación** | Validación cruzada (5-fold) + evaluación final en conjunto de prueba (20%) |
| **Modelo guardado** | Pipeline completo en `modelo/pipeline_modelo.joblib` |

### Conclusiones

1. El modelo fue entrenado y evaluado **evitando fuga de información**: se excluyó `final_grade` por derivarse del target, y todo el preprocesamiento se ajustó exclusivamente sobre el conjunto de entrenamiento.

2. Se compararon 6 modelos de regresión mediante validación cruzada, seleccionando el de mejor R².

3. Se identificaron las características más influyentes en el rendimiento académico a partir de la importancia de features del modelo.

4. El pipeline completo (preprocesamiento + modelo) fue guardado con `joblib`, listo para ser reutilizado en las siguientes fases del proyecto (scripts, Docker, API REST).